# `siglip2_server_v1.ipynb`

Kaggle server notebook for the repository **`zintomvn/Multimodal-Retrieval`**.

## Goal

Serve the repository's SigLIP2 text encoder as an OpenAI-compatible embedding API:

- `GET /healthz`
- `GET /readyz`
- `GET /v1/models`
- `POST /v1/embeddings`
- `GET /stats`

The web/backend can then use the existing retrieval flow:

```text
query
  ├─ OpenCLIP text embedding -> OpenCLIP Milvus collection -> ranked list A
  └─ SigLIP2 text embedding  -> SigLIP2  Milvus collection -> ranked list B
                                      ↓
                         visual RRF (repo: k = 60)
                                      ↓
                    hybrid metadata/text + final fusion
                                      ↓
                                   results
```

This notebook **does not mix OpenCLIP and SigLIP2 vectors directly**. They are different vector spaces. Each model searches its own collection and the repository performs rank fusion.

## Kaggle T4×2 serving design

The correct worker model here is not `uvicorn --workers N`.

- **1 FastAPI/Uvicorn process** owns the HTTP ingress.
- **2 model replicas**, one fixed to each T4 (`cuda:0`, `cuda:1`).
- One serial GPU executor per T4 prevents accidental concurrent forwards on the same model replica.
- Requests are **coalesced with dynamic batching** for a few milliseconds.
- Requests are distributed across both GPU queues.
- `fp16` is used on T4 to use Tensor Cores and reduce memory bandwidth/VRAM.
- `orjson`, `uvloop`, `httptools`, bounded queues and Uvicorn concurrency/backlog limits reduce CPU/network overhead and provide backpressure.

Creating many Uvicorn workers would duplicate model memory/process state and is the wrong way to scale a two-GPU notebook service.

## 0. Kaggle prerequisites

In Kaggle Notebook settings:

1. Accelerator: **GPU T4 ×2**
2. Internet: **ON**
3. For a public tunnel, create a Kaggle Secret named `SIGLIP2_API_KEY`.
4. Keep the notebook session running. Kaggle notebooks are not a production SLA hosting platform; the benchmark cells below measure the capacity of the current session.

The repository currently configures:

- model key: `siglip2_so400m16_384_webli_openclip_1152_v1`
- OpenCLIP model: `ViT-SO400M-16-SigLIP2-384`
- pretrained: `webli`
- dimension: `1152`
- collection: `keyframe_embeddings_siglip2_so400m16_384_webli_openclip_1152_v1`

This notebook reads those values from `configs/model_registry.yaml` instead of duplicating them in model-serving code.

In [ ]:
# 1. Clone/update the exact repository used by the web.
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/zintomvn/Multimodal-Retrieval.git"
REPO_DIR = Path("/kaggle/working/Multimodal-Retrieval")
REPO_REF = os.getenv("MULTIMODAL_RETRIEVAL_REF", "main")

if not REPO_DIR.exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)
    ])
else:
    subprocess.check_call(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_REF, "--depth", "1"])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "checkout", REPO_REF])
    subprocess.check_call(["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{REPO_REF}"])

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

print("repo:", REPO_DIR)
print("ref :", REPO_REF)
print("sha :", commit)

In [ ]:
# 2. Install serving dependencies.
#
# Important:
# The repository's current embedding requirements pin open_clip_torch==2.26.1,
# which predates SigLIP2 support. For this SigLIP2 server we intentionally
# install a modern OpenCLIP version while leaving Kaggle's existing torch build intact.

import subprocess
import sys

packages = [
    "open_clip_torch==3.3.0",
    "timm>=1.0.15,<2",
    "fastapi==0.115.12",
    "uvicorn[standard]==0.34.2",
    "httpx==0.28.1",
    "PyYAML==6.0.2",
    "orjson>=3.10,<4",
    "psutil>=5.9,<8",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages]
)
print("Dependencies installed.")

In [ ]:
# 3. Detect Kaggle CPU/GPU resources and apply runtime defaults.

import json
import multiprocessing
import os
import platform
import psutil
import torch

CPU_COUNT = os.cpu_count() or multiprocessing.cpu_count() or 1
GPU_COUNT = torch.cuda.device_count()

hardware = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cpu_count": CPU_COUNT,
    "ram_gb": round(psutil.virtual_memory().total / (1024**3), 2),
    "cuda_available": torch.cuda.is_available(),
    "gpu_count": GPU_COUNT,
    "gpus": [],
}

for i in range(GPU_COUNT):
    p = torch.cuda.get_device_properties(i)
    hardware["gpus"].append({
        "index": i,
        "name": p.name,
        "vram_gb": round(p.total_memory / (1024**3), 2),
        "compute_capability": f"{p.major}.{p.minor}",
    })

print(json.dumps(hardware, indent=2))

if GPU_COUNT < 1:
    raise RuntimeError("CUDA GPU not found. Enable a GPU accelerator in Kaggle.")

if GPU_COUNT < 2:
    print("WARNING: only one GPU is visible. The notebook will run, but T4×2 throughput is unavailable.")

# Avoid CPU thread oversubscription. The main CPU load is tokenization + JSON + networking.
# Each GPU has one dedicated inference thread; BLAS/OpenMP libraries should not spawn large pools.
PER_PROCESS_THREADS = max(1, min(2, CPU_COUNT // max(1, min(GPU_COUNT, 2))))
os.environ["OMP_NUM_THREADS"] = str(PER_PROCESS_THREADS)
os.environ["MKL_NUM_THREADS"] = str(PER_PROCESS_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

torch.set_num_threads(PER_PROCESS_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

print({
    "torch_num_threads": torch.get_num_threads(),
    "omp_num_threads": os.environ["OMP_NUM_THREADS"],
    "mkl_num_threads": os.environ["MKL_NUM_THREADS"],
})

In [ ]:
# 4. Load SigLIP2 configuration from the repository.

import yaml

SIGLIP2_MODEL_KEY = "siglip2_so400m16_384_webli_openclip_1152_v1"
REGISTRY_PATH = REPO_DIR / "configs" / "model_registry.yaml"
PROFILE_PATH = REPO_DIR / "configs" / "retrieval_profiles.yaml"

with REGISTRY_PATH.open("r", encoding="utf-8") as f:
    registry = yaml.safe_load(f) or {}

cfg = (registry.get("embedders") or {}).get(SIGLIP2_MODEL_KEY)
if not isinstance(cfg, dict):
    raise KeyError(f"{SIGLIP2_MODEL_KEY!r} not found in {REGISTRY_PATH}")

required = ["model", "openclip_model", "openclip_pretrained", "dim", "collection"]
missing = [k for k in required if cfg.get(k) in (None, "")]
if missing:
    raise ValueError(f"Missing required SigLIP2 config fields: {missing}")

MODEL_INFO = {
    "model_key": SIGLIP2_MODEL_KEY,
    "model_id": str(cfg["model"]).strip(),
    "openclip_model": str(cfg["openclip_model"]).strip(),
    "openclip_pretrained": str(cfg["openclip_pretrained"]).strip(),
    "expected_dim": int(cfg["dim"]),
    "collection": str(cfg["collection"]).strip(),
    "extractor_version": str(cfg.get("extractor_version", "")).strip(),
    "l2_normalize": bool(cfg.get("l2_normalize", True)),
}

print(json.dumps(MODEL_INFO, indent=2, ensure_ascii=False))

if MODEL_INFO["expected_dim"] != 1152:
    raise RuntimeError(
        f"Repository SigLIP2 dimension changed to {MODEL_INFO['expected_dim']}; "
        "verify the Milvus collection before serving."
    )

In [ ]:
# 5. Serving knobs.
#
# Defaults are aimed at text-query embedding on two 16-GB T4s.
# The load-test section is the source of truth for the current Kaggle session.

from dataclasses import dataclass

@dataclass(frozen=True)
class ServeConfig:
    host: str = "0.0.0.0"
    port: int = 8003

    # One model replica per GPU. On T4×2 this becomes [cuda:0, cuda:1].
    max_gpu_replicas: int = 2

    # Dynamic batching.
    gpu_batch_size: int = 64
    batch_wait_ms: float = 6.0

    # Backpressure and request limits.
    per_gpu_queue_max_texts: int = 4096
    max_texts_per_http_request: int = 256
    request_timeout_s: float = 30.0

    # Uvicorn ingress limits.
    uvicorn_limit_concurrency: int = 512
    uvicorn_backlog: int = 2048

    # Model precision.
    precision: str = "fp16"

SERVE = ServeConfig()

DEVICE_IDS = list(range(min(torch.cuda.device_count(), SERVE.max_gpu_replicas)))
if not DEVICE_IDS:
    raise RuntimeError("No CUDA devices available.")

print("GPU replicas:", [f"cuda:{i}" for i in DEVICE_IDS])
print("Serve config:", SERVE)

In [ ]:
import secrets

print(secrets.token_urlsafe(32))

In [ ]:
# 6. Load API key from Kaggle Secrets.
#
# Local access may run without a key.
# The public-tunnel cell later REFUSES to expose an unauthenticated endpoint.

def get_kaggle_secret(name: str) -> str:
    value = os.getenv(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return (UserSecretsClient().get_secret(name) or "").strip()
    except Exception:
        return ""

API_KEY = get_kaggle_secret("SIGLIP2_API_KEY")

print("SIGLIP2_API_KEY configured:", bool(API_KEY))
if not API_KEY:
    print(
        "Local server is allowed without auth. "
        "Add Kaggle Secret SIGLIP2_API_KEY before creating a public tunnel."
    )

In [ ]:
# 7. Validate that the installed OpenCLIP build knows the exact repo model/pretrained pair.

import open_clip

print("open_clip version:", getattr(open_clip, "__version__", "unknown"))

pretrained_pairs = set(open_clip.list_pretrained())
pair = (MODEL_INFO["openclip_model"], MODEL_INFO["openclip_pretrained"])

if pair not in pretrained_pairs:
    related = [
        item for item in sorted(pretrained_pairs)
        if "SigLIP2" in item[0] and "SO400M" in item[0]
    ]
    raise RuntimeError(
        "The exact repository model/pretrained pair is not registered by this OpenCLIP build.\n"
        f"wanted={pair}\n"
        f"related={related[:20]}"
    )

print("Verified OpenCLIP pretrained pair:", pair)

In [ ]:
# 8. Load one fp16 SigLIP2 text-encoder replica on each GPU and verify vector compatibility.

import gc
import math
import time
from dataclasses import dataclass

import torch.nn.functional as F

@dataclass
class LoadedReplica:
    gpu_id: int
    device: str
    model: object
    tokenizer: object

def load_replica(gpu_id: int) -> LoadedReplica:
    device = f"cuda:{gpu_id}"
    torch.cuda.set_device(gpu_id)

    print(f"[GPU {gpu_id}] loading {MODEL_INFO['openclip_model']} / {MODEL_INFO['openclip_pretrained']} ...")
    started = time.perf_counter()

    model, _, _ = open_clip.create_model_and_transforms(
        model_name=MODEL_INFO["openclip_model"],
        pretrained=MODEL_INFO["openclip_pretrained"],
        precision=SERVE.precision,
        device=device,
    )
    tokenizer = open_clip.get_tokenizer(MODEL_INFO["openclip_model"])
    model.eval()

    # Warm-up + compatibility probe.
    texts = [
        "a person wearing a red shirt in a crowded street",
        "a close-up of food being prepared in a kitchen",
    ]
    with torch.inference_mode():
        tokens = tokenizer(texts).to(device, non_blocking=True)
        vectors = model.encode_text(tokens).float()
        if MODEL_INFO["l2_normalize"]:
            vectors = F.normalize(vectors, dim=-1)
        torch.cuda.synchronize(gpu_id)

    if int(vectors.shape[-1]) != MODEL_INFO["expected_dim"]:
        raise RuntimeError(
            f"[GPU {gpu_id}] embedding dim mismatch: "
            f"{vectors.shape[-1]} != {MODEL_INFO['expected_dim']}"
        )

    norm = float(torch.linalg.vector_norm(vectors[0]).cpu())
    if MODEL_INFO["l2_normalize"] and abs(norm - 1.0) > 1e-2:
        raise RuntimeError(f"[GPU {gpu_id}] L2 norm check failed: {norm}")

    elapsed = time.perf_counter() - started
    free_b, total_b = torch.cuda.mem_get_info(gpu_id)
    used_gb = (total_b - free_b) / (1024**3)

    print(
        f"[GPU {gpu_id}] ready in {elapsed:.2f}s | dim={vectors.shape[-1]} "
        f"| norm={norm:.6f} | allocated+cached≈{used_gb:.2f} GiB"
    )
    return LoadedReplica(gpu_id, device, model, tokenizer)

REPLICAS = [load_replica(i) for i in DEVICE_IDS]

gc.collect()
torch.cuda.empty_cache()
print("Loaded replicas:", len(REPLICAS))

## 9. Dynamic batching engine

Why this matters for web search:

A normal synchronous embedding endpoint receives many `batch=1` requests. On GPU that wastes launch overhead and leaves throughput on the table. The engine below briefly waits (`batch_wait_ms`) and merges independent query texts into a larger GPU batch.

Each T4 has:

```text
async queue -> batch collector -> 1 dedicated inference thread -> its own model replica
```

The dispatcher sends new texts to the least-loaded GPU queue. Each GPU stays serial internally, while the two GPUs run in parallel.

In [ ]:
# 9. Dynamic batching engine.

import asyncio
import itertools
import threading
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from typing import Any

@dataclass
class PendingText:
    text: str
    future: asyncio.Future

@dataclass
class ReplicaStats:
    batches: int = 0
    texts: int = 0
    failures: int = 0
    total_infer_s: float = 0.0
    max_batch_seen: int = 0

class GPUWorker:
    def __init__(self, loaded: LoadedReplica):
        self.loaded = loaded
        self.queue: asyncio.Queue[PendingText] = asyncio.Queue(
            maxsize=SERVE.per_gpu_queue_max_texts
        )
        self.executor = ThreadPoolExecutor(
            max_workers=1,
            thread_name_prefix=f"siglip2-gpu-{loaded.gpu_id}",
        )
        self.task: asyncio.Task | None = None
        self.stats = ReplicaStats()

    async def start(self):
        if self.task is None or self.task.done():
            self.task = asyncio.create_task(
                self._batch_loop(),
                name=f"siglip2-batcher-gpu-{self.loaded.gpu_id}",
            )

    async def stop(self):
        if self.task and not self.task.done():
            self.task.cancel()
            try:
                await self.task
            except asyncio.CancelledError:
                pass
        self.executor.shutdown(wait=False, cancel_futures=True)

    def _encode_sync(self, texts: list[str]) -> list[list[float]]:
        gpu_id = self.loaded.gpu_id
        device = self.loaded.device
        model = self.loaded.model
        tokenizer = self.loaded.tokenizer

        torch.cuda.set_device(gpu_id)
        with torch.inference_mode():
            tokens = tokenizer(texts).to(device, non_blocking=True)
            vectors = model.encode_text(tokens).float()
            if MODEL_INFO["l2_normalize"]:
                vectors = F.normalize(vectors, dim=-1)
            # CPU transfer synchronizes this batch before it is returned.
            vectors_cpu = vectors.cpu()

        if int(vectors_cpu.shape[-1]) != MODEL_INFO["expected_dim"]:
            raise RuntimeError(
                f"GPU {gpu_id}: embedding dim {vectors_cpu.shape[-1]} "
                f"!= {MODEL_INFO['expected_dim']}"
            )
        return vectors_cpu.tolist()

    async def submit(self, item: PendingText):
        await self.queue.put(item)

    async def _batch_loop(self):
        loop = asyncio.get_running_loop()

        while True:
            first = await self.queue.get()
            batch = [first]

            deadline = loop.time() + SERVE.batch_wait_ms / 1000.0
            while len(batch) < SERVE.gpu_batch_size:
                remaining = deadline - loop.time()
                if remaining <= 0:
                    break
                try:
                    nxt = await asyncio.wait_for(self.queue.get(), timeout=remaining)
                    batch.append(nxt)
                except asyncio.TimeoutError:
                    break

            texts = [item.text for item in batch]
            started = time.perf_counter()

            try:
                vectors = await loop.run_in_executor(
                    self.executor, self._encode_sync, texts
                )
                elapsed = time.perf_counter() - started

                self.stats.batches += 1
                self.stats.texts += len(texts)
                self.stats.total_infer_s += elapsed
                self.stats.max_batch_seen = max(self.stats.max_batch_seen, len(texts))

                for item, vector in zip(batch, vectors):
                    if not item.future.done():
                        item.future.set_result(vector)
            except Exception as exc:
                self.stats.failures += 1
                for item in batch:
                    if not item.future.done():
                        item.future.set_exception(exc)
            finally:
                for _ in batch:
                    self.queue.task_done()

class EmbeddingPool:
    def __init__(self, loaded_replicas: list[LoadedReplica]):
        self.workers = [GPUWorker(r) for r in loaded_replicas]
        self.started = False
        self._rr = itertools.count()

    async def start(self):
        if self.started:
            return
        for worker in self.workers:
            await worker.start()
        self.started = True

    async def stop(self):
        for worker in self.workers:
            await worker.stop()
        self.started = False

    def _pick_worker(self) -> GPUWorker:
        # Prefer shortest queue. Round-robin breaks ties deterministically.
        depths = [w.queue.qsize() for w in self.workers]
        min_depth = min(depths)
        candidates = [i for i, d in enumerate(depths) if d == min_depth]
        idx = candidates[next(self._rr) % len(candidates)]
        return self.workers[idx]

    async def encode(self, texts: list[str]) -> list[list[float]]:
        loop = asyncio.get_running_loop()
        futures = []

        for text in texts:
            fut = loop.create_future()
            futures.append(fut)
            await self._pick_worker().submit(PendingText(text=text, future=fut))

        try:
            return await asyncio.wait_for(
                asyncio.gather(*futures),
                timeout=SERVE.request_timeout_s,
            )
        except asyncio.TimeoutError as exc:
            for fut in futures:
                if not fut.done():
                    fut.cancel()
            raise TimeoutError(
                f"Embedding request exceeded {SERVE.request_timeout_s}s"
            ) from exc

    def snapshot(self) -> dict[str, Any]:
        result = []
        for w in self.workers:
            s = w.stats
            avg_batch = (s.texts / s.batches) if s.batches else 0.0
            avg_infer_ms = (1000.0 * s.total_infer_s / s.batches) if s.batches else 0.0

            free_b, total_b = torch.cuda.mem_get_info(w.loaded.gpu_id)
            result.append({
                "gpu_id": w.loaded.gpu_id,
                "device": w.loaded.device,
                "queue_depth_texts": w.queue.qsize(),
                "queue_capacity_texts": SERVE.per_gpu_queue_max_texts,
                "batches": s.batches,
                "texts": s.texts,
                "failures": s.failures,
                "avg_batch_size": round(avg_batch, 2),
                "max_batch_seen": s.max_batch_seen,
                "avg_batch_infer_ms": round(avg_infer_ms, 3),
                "gpu_free_gb": round(free_b / (1024**3), 3),
                "gpu_total_gb": round(total_b / (1024**3), 3),
            })
        return {"workers": result}

POOL = EmbeddingPool(REPLICAS)
print("Embedding pool created with", len(POOL.workers), "GPU workers.")

In [ ]:
# 10. FastAPI app: OpenAI-compatible /v1/embeddings.

import hmac
from contextlib import asynccontextmanager
from typing import Annotated

from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.responses import ORJSONResponse
from pydantic import BaseModel, ConfigDict

class EmbeddingRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    model: str | None = None
    input: str | list[str]

class ServiceCounters:
    def __init__(self):
        self.requests = 0
        self.texts = 0
        self.errors = 0
        self.started_at = time.time()

COUNTERS = ServiceCounters()

def _as_texts(value: str | list[str]) -> list[str]:
    texts = [value] if isinstance(value, str) else [str(x) for x in value]
    texts = [t for t in texts if t is not None]
    return texts

def _require_auth(authorization: str | None) -> None:
    if not API_KEY:
        return
    expected = f"Bearer {API_KEY}"
    if not authorization or not hmac.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail="invalid bearer token")

@asynccontextmanager
async def lifespan(app: FastAPI):
    await POOL.start()
    yield
    await POOL.stop()

app = FastAPI(
    title="SigLIP2 Kaggle Embedding Service",
    version="1.0.0",
    default_response_class=ORJSONResponse,
    lifespan=lifespan,
)

@app.get("/healthz")
async def healthz():
    return {
        "status": "ok",
        "model_id": MODEL_INFO["model_id"],
        "model_key": MODEL_INFO["model_key"],
        "dim": MODEL_INFO["expected_dim"],
        "collection": MODEL_INFO["collection"],
        "gpu_replicas": len(POOL.workers),
        "auth_enabled": bool(API_KEY),
    }

@app.get("/readyz")
async def readyz():
    if not POOL.started:
        raise HTTPException(status_code=503, detail="embedding pool not started")
    return {"status": "ready", "gpu_replicas": len(POOL.workers)}

@app.get("/v1/models")
async def list_models(authorization: Annotated[str | None, Header()] = None):
    _require_auth(authorization)
    return {
        "object": "list",
        "data": [{
            "id": MODEL_INFO["model_id"],
            "object": "model",
            "owned_by": "kaggle-siglip2",
        }],
    }

@app.post("/v1/embeddings")
async def embeddings(
    body: EmbeddingRequest,
    authorization: Annotated[str | None, Header()] = None,
):
    _require_auth(authorization)

    texts = _as_texts(body.input)
    if not texts or any(not t.strip() for t in texts):
        raise HTTPException(status_code=400, detail="input must contain non-empty text")

    if len(texts) > SERVE.max_texts_per_http_request:
        raise HTTPException(
            status_code=413,
            detail=(
                f"too many texts in one HTTP request: {len(texts)} > "
                f"{SERVE.max_texts_per_http_request}"
            ),
        )

    if body.model and body.model != MODEL_INFO["model_id"]:
        raise HTTPException(
            status_code=400,
            detail=(
                f"model mismatch: requested {body.model!r}, "
                f"available {MODEL_INFO['model_id']!r}"
            ),
        )

    COUNTERS.requests += 1
    COUNTERS.texts += len(texts)

    try:
        vectors = await POOL.encode(texts)
    except TimeoutError as exc:
        COUNTERS.errors += 1
        raise HTTPException(status_code=504, detail=str(exc)) from exc
    except Exception as exc:
        COUNTERS.errors += 1
        raise HTTPException(status_code=500, detail=f"inference failed: {exc}") from exc

    data = [
        {"object": "embedding", "index": i, "embedding": vector}
        for i, vector in enumerate(vectors)
    ]

    # OpenCLIP tokenizer uses a fixed context tensor; this field is only compatibility metadata.
    return {
        "object": "list",
        "data": data,
        "model": MODEL_INFO["model_id"],
        "usage": {
            "prompt_tokens": 0,
            "total_tokens": 0,
        },
    }

@app.get("/stats")
async def stats(authorization: Annotated[str | None, Header()] = None):
    _require_auth(authorization)
    uptime = max(1e-9, time.time() - COUNTERS.started_at)
    return {
        "service": {
            "uptime_s": round(uptime, 2),
            "requests": COUNTERS.requests,
            "texts": COUNTERS.texts,
            "errors": COUNTERS.errors,
            "avg_requests_per_s_since_start": round(COUNTERS.requests / uptime, 3),
            "avg_texts_per_s_since_start": round(COUNTERS.texts / uptime, 3),
        },
        **POOL.snapshot(),
    }

print("FastAPI app ready.")

In [ ]:
# 11. Start a SINGLE Uvicorn ingress server in the notebook kernel.
import threading
import time
import httpx
import uvicorn

def stop_server():
    server = globals().get("SIGLIP2_UVICORN_SERVER")
    thread = globals().get("SIGLIP2_UVICORN_THREAD")
    if server is not None:
        server.should_exit = True
    if thread is not None and thread.is_alive():
        thread.join(timeout=10)

stop_server()

# Recreate async queues/executors so rerunning this cell is safe even though
# Uvicorn creates a fresh event loop for the new server thread.
POOL = EmbeddingPool(REPLICAS)
COUNTERS = ServiceCounters()

uv_config = uvicorn.Config(
    app,
    host=SERVE.host,
    port=SERVE.port,
    log_level="info",
    access_log=True,  # lower CPU/log overhead under load
    loop="uvloop",
    http="httptools",
    workers=1,
    limit_concurrency=SERVE.uvicorn_limit_concurrency,
    backlog=SERVE.uvicorn_backlog,
)

SIGLIP2_UVICORN_SERVER = uvicorn.Server(uv_config)
SIGLIP2_UVICORN_THREAD = threading.Thread(
    target=SIGLIP2_UVICORN_SERVER.run,
    daemon=True,
    name="siglip2-uvicorn",
)
SIGLIP2_UVICORN_THREAD.start()

LOCAL_ROOT = f"http://127.0.0.1:{SERVE.port}"
LOCAL_BASE_URL = f"{LOCAL_ROOT}/v1"

last_error = None
for _ in range(120):
    try:
        r = httpx.get(f"{LOCAL_ROOT}/healthz", timeout=2.0)
        if r.status_code == 200:
            print("Server:", LOCAL_BASE_URL)
            print(json.dumps(r.json(), indent=2))
            break
    except Exception as exc:
        last_error = exc
    time.sleep(0.5)
else:
    raise RuntimeError(f"Server did not become ready: {last_error}")

In [ ]:
# 12. Correctness check: model id, dimension, L2 normalization, and two-GPU service state.

import math

headers = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}

with httpx.Client(timeout=60.0, headers=headers) as client:
    models_resp = client.get(f"{LOCAL_BASE_URL}/models")
    models_resp.raise_for_status()

    payload = {
        "model": MODEL_INFO["model_id"],
        "input": [
            "a person wearing a red shirt in a crowded street",
            "two cyclists racing on a road",
        ],
    }
    emb_resp = client.post(f"{LOCAL_BASE_URL}/embeddings", json=payload)
    emb_resp.raise_for_status()
    body = emb_resp.json()

vectors = [item["embedding"] for item in body["data"]]
checks = []
for i, vector in enumerate(vectors):
    norm = math.sqrt(sum(float(x) * float(x) for x in vector))
    checks.append({"index": i, "dim": len(vector), "norm": round(norm, 6)})

if any(x["dim"] != MODEL_INFO["expected_dim"] for x in checks):
    raise RuntimeError(checks)
if MODEL_INFO["l2_normalize"] and any(abs(x["norm"] - 1.0) > 1e-2 for x in checks):
    raise RuntimeError(checks)

print(json.dumps({
    "ok": True,
    "model": body["model"],
    "checks": checks,
    "stats": httpx.get(f"{LOCAL_ROOT}/stats", headers=headers, timeout=10).json(),
}, indent=2))

## 13. Load test / capacity test

Do not guess a fixed RPS for Kaggle. Measure the live session.

The benchmark below reports:

- requests/second
- texts/second
- p50 / p95 / p99 HTTP latency
- failures

Start with `[1, 8, 16, 32, 64]` concurrency. If p95 starts growing rapidly without a meaningful throughput gain, the previous level is a better operating point.

Because the backend typically embeds short search queries, the default test sends `1 text/request`, which also exercises the dynamic batcher.

In [ ]:
# 13. Async load-test helpers.

import asyncio
import random
import statistics

SAMPLE_TEXTS = [
    "a person wearing a red shirt in a crowded street",
    "a chef preparing shrimp on a plate",
    "a close-up of mushrooms being sliced",
    "two cyclists racing on a road",
    "a lion dance performance on a city street",
    "people playing tug of war outdoors",
    "a man pouring water over his face",
    "a classroom presentation with text on a slide",
]

def percentile(values: list[float], q: float) -> float:
    if not values:
        return float("nan")
    ordered = sorted(values)
    idx = min(len(ordered) - 1, max(0, int(round((len(ordered) - 1) * q))))
    return ordered[idx]

async def benchmark(
    concurrency: int = 32,
    total_requests: int = 200,
    texts_per_request: int = 1,
) -> dict:
    semaphore = asyncio.Semaphore(concurrency)
    latencies = []
    failures = 0

    auth_headers = {"Authorization": f"Bearer {API_KEY}"} if API_KEY else {}
    limits = httpx.Limits(
        max_connections=max(concurrency * 2, 32),
        max_keepalive_connections=max(concurrency, 16),
    )

    async with httpx.AsyncClient(
        timeout=SERVE.request_timeout_s + 10,
        headers=auth_headers,
        limits=limits,
    ) as client:

        async def one(i: int):
            nonlocal failures
            texts = [
                SAMPLE_TEXTS[(i + j) % len(SAMPLE_TEXTS)]
                for j in range(texts_per_request)
            ]
            body_input = texts[0] if texts_per_request == 1 else texts
            payload = {"model": MODEL_INFO["model_id"], "input": body_input}

            async with semaphore:
                t0 = time.perf_counter()
                try:
                    r = await client.post(
                        f"{LOCAL_BASE_URL}/embeddings",
                        json=payload,
                    )
                    r.raise_for_status()
                    response = r.json()
                    if len(response.get("data", [])) != texts_per_request:
                        raise RuntimeError("unexpected embedding count")
                except Exception:
                    failures += 1
                finally:
                    latencies.append(time.perf_counter() - t0)

        started = time.perf_counter()
        await asyncio.gather(*(one(i) for i in range(total_requests)))
        wall = time.perf_counter() - started

    successful = total_requests - failures
    return {
        "concurrency": concurrency,
        "total_requests": total_requests,
        "texts_per_request": texts_per_request,
        "successful_requests": successful,
        "failures": failures,
        "wall_s": round(wall, 3),
        "requests_per_s": round(successful / wall, 3) if wall else 0.0,
        "texts_per_s": round(successful * texts_per_request / wall, 3) if wall else 0.0,
        "p50_ms": round(1000 * percentile(latencies, 0.50), 2),
        "p95_ms": round(1000 * percentile(latencies, 0.95), 2),
        "p99_ms": round(1000 * percentile(latencies, 0.99), 2),
    }

print("Benchmark helper ready.")

In [ ]:
# 14. Run a concurrency sweep.
#
# Change TOTAL_REQUESTS upward (e.g. 500-2000) for a more stable result after warm-up.

CONCURRENCY_LEVELS = [1, 8, 16, 32, 64]
TOTAL_REQUESTS = 200

benchmark_results = []
for concurrency in CONCURRENCY_LEVELS:
    result = await benchmark(
        concurrency=concurrency,
        total_requests=TOTAL_REQUESTS,
        texts_per_request=1,
    )
    benchmark_results.append(result)
    print(json.dumps(result, ensure_ascii=False))

print("\nGPU worker stats:")
print(json.dumps(
    httpx.get(f"{LOCAL_ROOT}/stats", headers=headers, timeout=10).json(),
    indent=2,
))

## 15. Public endpoint for your web/backend

Kaggle cannot be treated like a normal VM with a permanently routable inbound port. For a temporary development/competition endpoint, the cell below creates an outbound Cloudflare Quick Tunnel.

**Security rule:** the cell refuses to publish the service unless `SIGLIP2_API_KEY` exists.

Your backend should use:

```text
SIGLIP2_EMBEDDING_BASE_URL=https://<random>.trycloudflare.com/v1
SIGLIP2_API_KEY=<same secret configured in Kaggle>
```

And the SigLIP2 model entry in `configs/model_registry.yaml` should contain:

```yaml
api_key_env: SIGLIP2_API_KEY
```

The repository's OpenAI-compatible adapter already knows how to send `Authorization: Bearer ...`.

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("SIGLIP2_API_KEY")
print(secret_value_0)

In [ ]:
# 15. OPTIONAL: expose the local SigLIP2 service through a Cloudflare Quick Tunnel.
#
# Fixes the previous Kaggle failure:
# - The Quick Tunnel URL can be printed before Kaggle's DNS resolver can resolve it.
# - Therefore we do NOT immediately fail after receiving the URL.
# - We wait for cloudflared to register a tunnel connection.
# - Public DNS/health verification is retried and is non-fatal.
#
# Requirements:
#   1) Kaggle Internet = ON
#   2) SIGLIP2_API_KEY is configured
#   3) The local Uvicorn server is already running

import os
import queue
import re
import socket
import stat
import subprocess
import threading
import time
import urllib.request
from pathlib import Path
from urllib.parse import urlparse

import httpx


CLOUDFLARED_BIN = Path("/kaggle/working/cloudflared")
CLOUDFLARE_URL_RE = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")


def stop_cloudflared() -> None:
    """Stop a previously started cloudflared process in this notebook kernel."""
    proc = globals().get("CLOUDFLARED_PROCESS")
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)


def install_cloudflared() -> Path:
    """Download the current cloudflared Linux amd64 binary once."""
    if CLOUDFLARED_BIN.exists():
        # Make sure it is executable even if the file came from a previous run.
        CLOUDFLARED_BIN.chmod(CLOUDFLARED_BIN.stat().st_mode | stat.S_IEXEC)
        return CLOUDFLARED_BIN

    download_url = (
        "https://github.com/cloudflare/cloudflared/releases/latest/download/"
        "cloudflared-linux-amd64"
    )
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(download_url, CLOUDFLARED_BIN)
    CLOUDFLARED_BIN.chmod(CLOUDFLARED_BIN.stat().st_mode | stat.S_IEXEC)

    version = subprocess.run(
        [str(CLOUDFLARED_BIN), "--version"],
        text=True,
        capture_output=True,
        check=True,
    )
    print(version.stdout.strip() or version.stderr.strip())
    return CLOUDFLARED_BIN


def verify_local_origin(local_root: str) -> None:
    """Fail early only when the actual local SigLIP2 server is unavailable."""
    url = f"{local_root.rstrip('/')}/healthz"
    response = httpx.get(url, timeout=10.0)
    response.raise_for_status()
    body = response.json()
    if body.get("status") != "ok":
        raise RuntimeError(f"Local SigLIP2 server is not healthy: {body}")
    print("Local origin healthy:", url)


def _stream_process_output(proc: subprocess.Popen, out_queue: queue.Queue) -> None:
    """Read cloudflared output without blocking the notebook's timeout loop."""
    if proc.stdout is None:
        return
    for line in iter(proc.stdout.readline, ""):
        out_queue.put(line.rstrip("\n"))
    out_queue.put(None)


def start_cloudflare_tunnel(local_root: str, startup_timeout_s: float = 90.0) -> str:
    """
    Start a Cloudflare Quick Tunnel and return https://<random>.trycloudflare.com.

    Important:
    Receiving the hostname and DNS resolution are separate events. We only require
    cloudflared to remain alive and establish a tunnel connection here.
    """
    if not API_KEY:
        raise RuntimeError(
            "Refusing to expose an unauthenticated embedding service. "
            "Create Kaggle Secret SIGLIP2_API_KEY, then rerun the secret/server cells."
        )

    verify_local_origin(local_root)
    binary = install_cloudflared()
    stop_cloudflared()

    # Let cloudflared choose the best transport instead of forcing HTTP/2.
    # Quick Tunnel usage follows Cloudflare's documented:
    #   cloudflared tunnel --url http://localhost:<port>
    cmd = [
        str(binary),
        "tunnel",
        "--no-autoupdate",
        "--url",
        local_root,
        "--loglevel",
        "info",
    ]

    print("Starting Cloudflare Quick Tunnel...")
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    globals()["CLOUDFLARED_PROCESS"] = proc

    output_queue: queue.Queue = queue.Queue()
    reader = threading.Thread(
        target=_stream_process_output,
        args=(proc, output_queue),
        daemon=True,
    )
    reader.start()

    public_root = None
    registered = False
    logs: list[str] = []
    deadline = time.monotonic() + startup_timeout_s

    while time.monotonic() < deadline:
        if proc.poll() is not None:
            break

        try:
            line = output_queue.get(timeout=0.25)
        except queue.Empty:
            continue

        if line is None:
            break

        logs.append(line)

        match = CLOUDFLARE_URL_RE.search(line)
        if match and public_root is None:
            public_root = match.group(0)
            print("Tunnel hostname created:", public_root)

        # A URL may be printed slightly before DNS is visible to the notebook.
        # Waiting for a registered connection gives a stronger readiness signal.
        lower = line.lower()
        if "registered tunnel connection" in lower:
            registered = True
            if public_root:
                break

    if proc.poll() is not None:
        tail = "\n".join(logs[-40:])
        raise RuntimeError(
            "cloudflared exited before the tunnel became usable.\n"
            f"Exit code: {proc.returncode}\n{tail}"
        )

    if not public_root:
        tail = "\n".join(logs[-40:])
        raise RuntimeError(
            "cloudflared is running but no trycloudflare.com URL was found.\n"
            f"{tail}"
        )

    if not registered:
        # Do not immediately kill a tunnel only because this particular log text
        # was not observed. The process is still alive and the URL was issued.
        print(
            "Warning: URL was issued, but the 'Registered tunnel connection' log "
            "was not observed before the startup timeout."
        )
    else:
        print("Cloudflare tunnel connection registered.")

    return public_root


def wait_for_public_health(
    public_root: str,
    timeout_s: float = 120.0,
    poll_s: float = 2.0,
) -> bool:
    """
    Wait for public DNS + /healthz, but DO NOT destroy a valid tunnel just because
    Kaggle's resolver is slow.

    Returns True when Kaggle can reach the public URL, otherwise False.
    """
    parsed = urlparse(public_root)
    hostname = parsed.hostname
    if not hostname:
        print("Warning: could not parse public hostname:", public_root)
        return False

    deadline = time.monotonic() + timeout_s
    last_error = None
    dns_announced = False

    while time.monotonic() < deadline:
        proc = globals().get("CLOUDFLARED_PROCESS")
        if proc is None or proc.poll() is not None:
            raise RuntimeError(
                "cloudflared stopped while waiting for the public endpoint."
            )

        # Step 1: DNS. This was the exact failure in the previous notebook.
        try:
            addresses = socket.getaddrinfo(
                hostname,
                443,
                type=socket.SOCK_STREAM,
                proto=socket.IPPROTO_TCP,
            )
            if addresses and not dns_announced:
                ips = sorted({item[4][0] for item in addresses})
                print("Public DNS resolved:", hostname, "->", ", ".join(ips[:4]))
                dns_announced = True
        except OSError as exc:
            last_error = f"DNS not ready: {exc}"
            time.sleep(poll_s)
            continue

        # Step 2: HTTP health through the tunnel.
        try:
            response = httpx.get(
                f"{public_root}/healthz",
                timeout=10.0,
                follow_redirects=True,
            )
            if response.status_code == 200:
                body = response.json()
                print("Public health check OK:", body)
                return True
            last_error = f"HTTP {response.status_code}: {response.text[:300]}"
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"

        time.sleep(poll_s)

    print(
        "WARNING: Kaggle could not verify the public hostname before timeout.\n"
        f"Last error: {last_error}\n"
        "The tunnel process is still running and the URL is still kept. "
        "Try the printed URL from your backend/local machine; Kaggle DNS can lag "
        "behind creation of a new trycloudflare.com hostname."
    )
    return False


# Set True when you want to expose the server.
CREATE_PUBLIC_TUNNEL = True

if CREATE_PUBLIC_TUNNEL:
    PUBLIC_ROOT = start_cloudflare_tunnel(LOCAL_ROOT)
    PUBLIC_BASE_URL = f"{PUBLIC_ROOT}/v1"

    print()
    print("=" * 72)
    print("PUBLIC_ROOT     =", PUBLIC_ROOT)
    print("PUBLIC_BASE_URL =", PUBLIC_BASE_URL)
    print("=" * 72)
    print()

    # This check retries DNS instead of immediately crashing with:
    #   ConnectError [Errno -2] Name or service not known
    PUBLIC_ENDPOINT_VERIFIED_FROM_KAGGLE = wait_for_public_health(
        PUBLIC_ROOT,
        timeout_s=120,
        poll_s=2,
    )

    print()
    print("Backend environment:")
    print(f"SIGLIP2_EMBEDDING_BASE_URL={PUBLIC_BASE_URL}")
    print("SIGLIP2_API_KEY=<same value as the Kaggle secret>")
else:
    PUBLIC_ROOT = None
    PUBLIC_BASE_URL = None
    PUBLIC_ENDPOINT_VERIFIED_FROM_KAGGLE = False
    print("Public tunnel not started. Set CREATE_PUBLIC_TUNNEL = True to enable it.")


## 16. Enable SigLIP2 in the current repository retrieval pipeline

The repository already has the correct model key, Milvus collection and visual RRF configuration, but SigLIP2 is currently disabled.

On the machine running your FastAPI backend:

### Environment

```bash
SIGLIP2_EMBEDDING_BASE_URL=https://<your-tunnel>.trycloudflare.com/v1
SIGLIP2_API_KEY=<same-secret>
```

### `configs/model_registry.yaml`

For `siglip2_so400m16_384_webli_openclip_1152_v1`:

```yaml
provider: openai_compatible
base_url: http://localhost:8003/v1  # env overrides this
api_key_env: SIGLIP2_API_KEY
model: ViT-SO400M-16-SigLIP2-384-webli
dim: 1152
l2_normalize: true
collection: keyframe_embeddings_siglip2_so400m16_384_webli_openclip_1152_v1
enabled: true
```

### `configs/retrieval_profiles.yaml`

For `competition_default`:

```yaml
visual_rrf:
  enabled: true
  k: 60

visual_models:
  clip_global:
    model_key: clip_vith14_quickgelu_dfn5b_v2
    collection: keyframe_embeddings_clip_vith14_quickgelu_dfn5b_v2
    weight: 1.0
    enabled: true

  siglip2_fine_grained:
    model_key: siglip2_so400m16_384_webli_openclip_1152_v1
    collection: keyframe_embeddings_siglip2_so400m16_384_webli_openclip_1152_v1
    weight: 1.0
    enabled: true
```

The important invariant is:

```text
SigLIP2 query embedding model
==
SigLIP2 model used to build the stored 1152-d image embeddings
==
SigLIP2 Milvus collection configured in retrieval profile
```

If any of those differ, similarity search is invalid even if the dimensions happen to match.

In [ ]:
# 16. Print backend integration values after a tunnel has been created.

if PUBLIC_BASE_URL:
    print("Backend environment:")
    print(f"SIGLIP2_EMBEDDING_BASE_URL={PUBLIC_BASE_URL}")
    print("SIGLIP2_API_KEY=<same Kaggle secret>")
    print()
    print("model_registry.yaml SigLIP2 fields to verify:")
    print(
        yaml.safe_dump(
            {
                SIGLIP2_MODEL_KEY: {
                    "provider": "openai_compatible",
                    "base_url": "http://localhost:8003/v1",
                    "api_key_env": "SIGLIP2_API_KEY",
                    "model": MODEL_INFO["model_id"],
                    "dim": MODEL_INFO["expected_dim"],
                    "l2_normalize": MODEL_INFO["l2_normalize"],
                    "collection": MODEL_INFO["collection"],
                    "enabled": True,
                }
            },
            sort_keys=False,
        )
    )
else:
    print("Create the public tunnel first to print the final backend URL.")

## 17. Operational notes

### Why only one Uvicorn worker?

`uvicorn --workers 2/4/...` means multiple Python **processes**. If each process initializes both GPUs, model replicas are duplicated and GPU memory/CPU initialization are wasted. The notebook instead keeps HTTP concurrency async and scales GPU inference explicitly with two device-bound replicas.

### Why dynamic batching?

For search traffic, many users issue short independent text queries. A small batch window can turn many batch-1 forwards into batch-8/16/32/64 forwards. This normally raises GPU utilization and throughput, at the cost of a few milliseconds of queueing latency.

### How to tune

Use the live benchmark rather than a guessed worker number.

- If GPU utilization is low and p95 is small: increase concurrency.
- If average batch size stays tiny: increase `batch_wait_ms` slightly (e.g. 6 → 8 ms).
- If p95 grows sharply while RPS stops improving: reduce client concurrency.
- If T4 memory has a large margin: test `gpu_batch_size=96` or `128`.
- If OOM occurs: reduce `gpu_batch_size` to `32`.
- For a competition/demo with a few simultaneous users, prefer stable p95 over maximum synthetic RPS.

### Kaggle limitation

This is a high-throughput **notebook-hosted inference service**, not a durable production deployment. The URL disappears when the Kaggle session/tunnel stops. For a permanent endpoint, keep the same server architecture but move it to a persistent GPU host.

In [ ]:
# # 18. Shutdown helpers (run manually when needed).

# def shutdown_all():
#     stop_cloudflared()
#     stop_server()
#     print("SigLIP2 server and tunnel stopped.")

# # shutdown_all()